In [ ]:
import itertools
import ROOT as R
import math
import csv
import os

# %jsroot on
# %matplotlib widget
R.EnableImplicitMT()

# Data
# data_files = "~/hps/track_cluster_matching/data/*.root"
# df_data = R.RDataFrame("MiniDST", data_files)
# print("Events in data:", df_data.Count().GetValue())

# Montecarlo
mc_files = "/net/data/pumpkin/HPS/physrun2021/sim/hpsForward_e-_3.742GeV*.root"
df_mc = R.RDataFrame("MiniDST", mc_files)
print("Events in MC:", df_mc.Count().GetValue())

In [ ]:
# Plot mean or sum energy maps in x-y bins, for data and MC, side by side.

def mean_energy_map(df, name, title,
                    nx=120, xmin=-300, xmax=300,
                    ny=60,  ymin=-150, ymax=150):
    """
    Returns (h_mean, h_sumE, h_count) where:
      h_sumE(x,y)  = sum of cluster energies in bin
      h_count(x,y) = number of clusters in bin
      h_mean(x,y)  = h_sumE / h_count
    Works with vector branches: ecal_cluster_x/y/energy (RVec).
    """

    # weight=1 vector with same length as clusters, so Histo2D can count entries
    df2 = df.Define("clus_one", "ROOT::VecOps::RVec<float>(ecal_cluster_energy.size(), 1.0f)")

    h_sumE = df2.Histo2D(
        (f"h_sumE_{name}", f"{title}; ECAL x [mm]; ECAL y [mm]", nx, xmin, xmax, ny, ymin, ymax),
        "ecal_cluster_x", "ecal_cluster_y", "ecal_cluster_energy"
    )

    h_cnt = df2.Histo2D(
        (f"h_cnt_{name}", f"{title} (counts); ECAL x [mm]; ECAL y [mm]", nx, xmin, xmax, ny, ymin, ymax),
        "ecal_cluster_x", "ecal_cluster_y", "clus_one"
    )

    # Trigger event loop once
    hs = h_sumE.GetPtr()
    hc = h_cnt.GetPtr()

    h_mean = hs.Clone(f"h_meanE_{name}")
    h_mean.SetTitle(f"{title} (mean cluster energy); ECAL x [mm]; ECAL y [mm]")
    h_mean.Divide(hc)  # mean = sum / count
    return h_mean, hs, hc


# --- build maps ---
h_mean_data, h_sum_data, h_cnt_data = mean_energy_map(df_data, "data", "DATA")
h_mean_mc,   h_sum_mc,   h_cnt_mc   = mean_energy_map(df_mc,   "mc",   "FEE MC")

# --- draw side-by-side ---
c = R.TCanvas("c_ecal_maps", "ECAL mean energy maps", 1400, 650)
c.Divide(2,1)

c.cd(1); R.gPad.SetGrid()
h_mean_data.Draw("COLZ")
# h_sum_data.Draw("COLZ")

c.cd(2); R.gPad.SetGrid()
h_mean_mc.Draw("COLZ")
# h_sum_mc.Draw("COLZ")
c.Draw()
# c.SaveAs("plots/ecal_mean_energy_maps_data_vs_mc.png")

In [ ]:
# Create list of all seed crystals (ix, iy)
# ------------------------------------------------------

# Get min and max ix, iy
ix_min = int(df_mc.Min("ecal_cluster_seed_ix").GetValue())  # -23
ix_max = int(df_mc.Max("ecal_cluster_seed_ix").GetValue())  #  23
iy_min = int(df_mc.Min("ecal_cluster_seed_iy").GetValue())  # -5
iy_max = int(df_mc.Max("ecal_cluster_seed_iy").GetValue())  #  5

# Create arrays of ix and iy values based on the min and max from MC
ix_arr = list(range(ix_min, ix_max + 1))
iy_arr = list(range(iy_min, iy_max + 1))


def is_valid_crystal(x, y):
    if y == 0:
        return False
    if x == 0:
        return False
    if -10 < x < -1 and y == 1:
        return False
    if -10 < x < -1 and y == -1:
        return False
    if x == 3 and y == 5:
        return False
    if x == -18 and y == 5:
        return False
    if x == -1 and y == -5:
        return False
    return True


all_pairs = list(itertools.product(ix_arr, iy_arr))
pairs = [
    (x, y) for (x, y) in itertools.product(ix_arr, iy_arr) if is_valid_crystal(x, y)
]
excluded = [(x, y) for (x, y) in all_pairs if not is_valid_crystal(x, y)]

print(f"Total possible crystals: {len(all_pairs)}")
print(f"Total valid crystals: {len(pairs)}")
print(f"Exculded: {len(excluded)}")
print("First 10:", pairs[:10])

In [ ]:
E_BEAM = 3.742  # GeV

# CB Function
R.gInterpreter.Declare("""
double crystalBall(double x, double norm, double mu, double sigma, double alpha, double n) {
    double t = (x - mu) / sigma;
    if (t > -alpha) {
        return norm * std::exp(-0.5 * t * t);   // Gaussian core
    } else {
        double A = std::pow(n / alpha, n) * std::exp(-0.5 * alpha * alpha);
        double B = n / alpha - alpha;
        return norm * A * std::pow(B - t, -n);  // power-law tail
    }
}
""")

CB_FORMULA = "crystalBall(x, [0], [1], [2], [3], [4])"


def fit_fee_peak_for_seed(df, seed_ix, seed_iy, tag,
                          Emin=2.0, seedFracMin=0.6,
                          pmin=0.3, ep_window=0.25,
                          nbins=120, xlo=1.5, xhi=4.5,
                          fit_halfwidth=0.30, min_entries=1,
                          fit_type="gaus",
                          plot_dir="plots"):

    os.makedirs(plot_dir, exist_ok=True)

    d = (df
        .Filter("ecal_cluster_energy.size() > 0 && part_pdg.size() > 0")
        .Define("seed_over_etot",
                "ecal_cluster_seed_energy / (ecal_cluster_energy + 1e-9f)")
        .Define("Esel", f"""
            ROOT::VecOps::RVec<float> out;
            for (int i = 0; i < (int)part_pdg.size(); ++i) {{
                int tr = part_track[i];
                int cl = part_ecal_cluster[i];
                if (tr < 0 || cl < 0) continue;                                 // Make sure to have macthed track and cluster
                
                if (ecal_cluster_seed_ix[cl] != {seed_ix} ||
                    ecal_cluster_seed_iy[cl] != {seed_iy}) continue;            // Look at required crystal
                
                float eclus = ecal_cluster_energy[cl];
                if (eclus < {Emin}f) continue;                                  // Min. cluster energy check
                
                float seedfrac = seed_over_etot[cl];
                if (seedfrac < {seedFracMin}f) continue;                        // Min. seed to total energy fraction check
                
                if (part_pdg[i] != 11) continue;                                // choose only electrons
                
                float px = track_px[tr], py = track_py[tr], pz = track_pz[tr];
                float psum = std::sqrt(px*px + py*py + pz*pz);
                if (psum < {pmin}f) continue;                                   // Min PSum check
                
                float ep = eclus / (psum + 1e-9f);
                if (std::abs(ep - 1.0f) > {ep_window}f) continue;               // track p and cluster energy difference check
                
                out.push_back(eclus);
            }}
            return out;
        """)
        .Filter("Esel.size() > 0")
    )

    hR = d.Histo1D((f"hE_{tag}_{seed_ix}_{seed_iy}",
                    f"Eclus seed({seed_ix},{seed_iy}), {fit_type} fit [{tag}]; E [GeV]; Counts",
                    nbins, xlo, xhi), "Esel")
    h = hR.GetPtr()

    peak_bin = h.GetMaximumBin()
    peak_x   = h.GetXaxis().GetBinCenter(peak_bin)
    fname    = f"f_{tag}_{seed_ix}_{seed_iy}"

    if fit_type == "cb":
        f = R.TF1(fname, CB_FORMULA,
                  peak_x - (2*fit_halfwidth), peak_x + fit_halfwidth)
        f.SetParameters(h.GetMaximum(), peak_x, 0.05, 1.2, 2.0)
        f.SetParNames("Norm", "mu", "sigma", "alpha", "n")
    else:
        f = R.TF1(fname, "gaus",
                  peak_x - fit_halfwidth, peak_x + fit_halfwidth)

    ret = int(h.Fit(f, "RQ0"))
    n   = int(h.GetEntries())
    mu  = float(f.GetParameter(1))
    sig = abs(float(f.GetParameter(2)))
    nDoF = int(f.GetNDF())
    chi2_ndf = f.GetChisquare() / f.GetNDF() if f.GetNDF() > 0 else -1.0
    # print(f"seed({seed_ix},{seed_iy}) | entries={n} | mu={mu:.4f} \
    # | sig={sig:.4f} | chi2={chi2_ndf:.4f} | nDof={nDoF} | ret={ret}")

    # --- sanity guards ---
    if n < min_entries:
        # print(f"seed({seed_ix},{seed_iy}) | SKIP: too few entries ({n})")
        return None

    if not (E_BEAM * 0.5 < mu < E_BEAM * 1.1):
        # print(f"seed({seed_ix},{seed_iy}) | SKIP: unphysical mu={mu:.4f} GeV")
        return None

    # --- per-crystal plot ---
    c = R.TCanvas(f"c_{tag}_{seed_ix}_{seed_iy}", "", 700, 500)
    c.SetGrid()

    h.SetLineColor(R.kBlue + 1)
    h.SetLineWidth(2)
    h.SetStats(0)
    h.Draw("HIST")
 
    f.SetLineColor(R.kRed)
    f.SetLineWidth(2)
    f.Draw("SAME")

    label = R.TLatex()
    label.SetNDC()
    label.SetTextSize(0.035)
    label.DrawLatex(0.15, 0.85, f"seed ({seed_ix}, {seed_iy})  [{tag}]")
    label.DrawLatex(0.15, 0.80, f"#mu = {mu:.4f} GeV")
    label.DrawLatex(0.15, 0.75, f"#sigma = {sig:.4f} GeV")
    label.DrawLatex(0.15, 0.70, f"#mu/E_{{beam}} = {mu/E_BEAM:.4f}")
    label.DrawLatex(0.15, 0.65, f"N = {n}")
    label.DrawLatex(0.15, 0.55, f"#chi^{{2}}/ndf = {chi2_ndf:.2f}")
    label.DrawLatex(0.15, 0.50, f"nDoF = {nDoF}")

    plot_path = os.path.join(plot_dir, f"fee_{tag}_ix{seed_ix}_iy{seed_iy}.png")
    c.SaveAs(plot_path)
    c.Close()

    return mu, sig, n, chi2_ndf, nDoF, ret

def make_fee_mc_products(df_mc, pairs, fit_type="cb", nbins=120, out_dir="fee_mc_output"):

    os.makedirs(out_dir, exist_ok=True)

    out_csv  = os.path.join(out_dir, f"fee_mc_peaks_{fit_type}.csv")
    out_root = os.path.join(out_dir, f"fee_mc_maps_{fit_type}.root")
    out_png  = os.path.join(out_dir, f"fee_mc_muOverEbeam_{fit_type}.png")
    plot_dir = os.path.join(out_dir, f"{fit_type}_plots")

    os.makedirs(plot_dir, exist_ok=True)

    h_muOverE = R.TH2D("h_muOverEbeam",
                       "FEE MC peak #mu / E_{beam}; seed ix; seed iy",
                       47, -23.5, 23.5,
                       11, -5.5, 5.5)

    results = []  # Store good fits
    bad = []      # To store the bad/no fit pairs

    for ix, iy in pairs:
        r = fit_fee_peak_for_seed(df_mc, ix, iy, "MC",
                                  fit_type=fit_type,
                                  plot_dir=plot_dir,
                                  nbins=nbins)
        if r is None:
            bad.append((ix, iy))
            continue

        mu, sig, n, chi2_ndf, nDoF, ret = r
        results.append((ix, iy, n, mu, sig, chi2_ndf, nDoF, mu / E_BEAM, ret))
        h_muOverE.SetBinContent(h_muOverE.GetXaxis().FindBin(ix),
                                h_muOverE.GetYaxis().FindBin(iy),
                                mu / E_BEAM)

    with open(out_csv, "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["seed_ix", "seed_iy", "n", "mu_GeV", "sigma_GeV",
                    "chi2_ndf", "DoF", "mu_over_Ebeam", "fit_status"])
        w.writerows(results)

    fout = R.TFile(out_root, "RECREATE")
    h_muOverE.Write()
    fout.Close()

    c = R.TCanvas("c_fee_mc", "FEE MC mu/Ebeam", 900, 450)
    R.gPad.SetRightMargin(0.14)
    h_muOverE.SetStats(0)
    h_muOverE.Draw("COLZ")
    
    # draw E_BEAM reference line info
    label = R.TLatex()
    label.SetNDC()
    label.SetTextSize(0.035)
    label.DrawLatex(0.12, 0.92, f"FEE MC  |  fit: {fit_type.upper()}  |  nbins: {nbins}  |  {len(results)} crystals")
    
    c.Update()
    c.Draw()
    c.SaveAs(out_png)

    # Write bad pairs in a file
    bad_txt = os.path.join(out_dir, f"bad_{fit_type}.txt")
    with open(bad_txt, "w") as f:
        f.write(f"Total bad pairs: {len(bad)}\n")
        f.write(str(bad) + "\n")

    print(f"[OK] Wrote {bad_txt} with {len(bad)} bad crystals.")
    
    return results, bad  # return both

In [ ]:
FIT_TYPE = "cb"
NBINS = 120

# pairs_ = [(4, -2), (3, 2), (10, 2), (-2, 2), (-13, 2), (-13, -2), (-18, -2), (3, -2), (6, -2), (11, -2)]
pairs_ = pairs

results_cb, bad_cb = make_fee_mc_products(df_mc, 
                                          pairs_, 
                                          fit_type=FIT_TYPE, 
                                          nbins=NBINS,
                                          out_dir=f"3rdrun_{NBINS}nbins_with_min_entries_1")

In [ ]:
FIT_TYPE = "cb"
NBINS = 120

# pairs_ = [(4, -2), (3, 2), (10, 2), (-2, 2), (-13, 2), (-13, -2), (-18, -2), (3, -2), (6, -2), (11, -2)]
pairs_ = pairs

results_cb, bad_cb = make_fee_mc_products(df_mc, pairs_,
                                          fit_type=FIT_TYPE,
                                          nbins=NBINS,
                                          out_dir=f"nbins_{NBINS}_no_trk_req")